In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Load Data
df = pd.read_csv('loan_approval_dataset.csv')

# 2. Critical Data Cleaning: Strip invisible whitespaces from column names
df.columns = df.columns.str.strip()

# Strip whitespaces from all string (object) columns
for col in df.select_dtypes(include='object'):
    df[col] = df[col].str.strip()

# 3. Target Encoding
# Convert 'Approved' to 1, 'Rejected' to 0
df['loan_status'] = (df['loan_status'] == 'Approved').astype(int)

# Drop loan_id as it has no predictive value
X = df.drop(columns=['loan_id', 'loan_status'])
y = df['loan_status']

# 4. Pipeline Setup
cat_features = ['education', 'self_employed']
num_features = ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 
                'cibil_score', 'residential_assets_value', 'commercial_assets_value', 
                'luxury_assets_value', 'bank_asset_value']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Transform Data
X_train_tf = preprocessor.fit_transform(X_train)
X_test_tf = preprocessor.transform(X_test)

# 6. Train XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train_tf, y_train)

# 7. Evaluate
y_pred = xgb_model.predict(X_test_tf)

print("--- Automated Underwriting Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 8. Serialize Assets
joblib.dump(preprocessor, "underwriting_preprocessor.joblib")
joblib.dump(xgb_model, "xgboost_underwriting_model.joblib")
print("\nPipeline and Model saved successfully.")